# 🧪 Test Loader Pipeline — 1 cặp En↔Vi

Notebook này minh họa từng bước một cặp câu đi qua toàn bộ pipeline trong `loader.py`:

1. `normalize_unicode` — chuẩn hoá NFC
2. `clean_text` — làm sạch noise (HTML, URL, subtitle, ...)
3. `is_valid_pair` — lọc cặp không hợp lệ
4. `load_dataset` — load toàn bộ file CSV (thực chiến)
5. `TranslationDataset.__getitem__` — tokenize + tensor
6. `collate_fn` — batch & padding

In [1]:
# ── Setup path để import src ──
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # gốc project

from src.data.loader import normalize_unicode, clean_text, is_valid_pair, load_dataset
print("✅ Import OK")

✅ Import OK


## 1. Định nghĩa cặp câu mẫu

Chúng ta tạo vài cặp câu để test đủ các trường hợp.

In [2]:
test_pairs = [
    # --- Cặp 1: Câu bình thường ---
    {
        "label": "Bình thường",
        "en": "Hello, how are you?",
        "vi": "Xin chào, bạn có khoẻ không?",
    },
    # --- Cặp 2: Có subtitle noise ---
    {
        "label": "Subtitle noise: [Muttering], ♪, --, CHAVEZ:",
        "en": "[Muttering] - Oh, I see ♪ ---- hello ----",
        "vi": "CHAVEZ: >> Ừ, tôi thấy rồi.",
    },
    # --- Cặp 3: Có HTML + URL ---
    {
        "label": "HTML + URL",
        "en": "Visit <b>our</b> site at https://example.com for more info.",
        "vi": "Truy cập trang web của <b>chúng tôi</b> tại www.example.vn.",
    },
    # --- Cặp 4: Câu lặp từ ---
    {
        "label": "Lặp từ (ha ha ha ha ha)",
        "en": "Ha ha ha ha ha, that was funny.",
        "vi": "Ha ha ha ha ha, cái đó buồn cười thật.",
    },
    # --- Cặp 5: Sẽ bị TỪ CHỐI (câu en == vi) ---
    {
        "label": "❌ Bị lọc: en == vi",
        "en": "OK",
        "vi": "OK",
    },
    # --- Cặp 6: Sẽ bị TỪ CHỐI (toàn dấu ...) ---
    {
        "label": "❌ Bị lọc: toàn dấu chấm lửng",
        "en": ".......",
        "vi": "Xin chào.",
    },
    # --- Cặp 7: Unicode tổ hợp dấu tiếng Việt ---
    {
        "label": "Unicode NFC (dấu tổ hợp → 1 codepoint)",
        "en": "The weather is nice today.",
        # 'ạ' viết dạng 2 codepoint (a + combining dot below)
        "vi": "Thời tiết hôm nay rất đe\u0323p.",
    },
]

print(f"Có {len(test_pairs)} cặp test")

Có 7 cặp test


## 2. Step-by-step: normalize_unicode → clean_text → is_valid_pair

In [3]:
for pair in test_pairs:
    en_raw = pair["en"]
    vi_raw = pair["vi"]
    label  = pair["label"]

    print("=" * 70)
    print(f"📌 [{label}]")
    print(f"   EN raw : {repr(en_raw)}")
    print(f"   VI raw : {repr(vi_raw)}")

    # Bước 1: normalize unicode
    en_nfc = normalize_unicode(en_raw)
    vi_nfc = normalize_unicode(vi_raw)
    print(f"   EN nfc : {repr(en_nfc)}")
    print(f"   VI nfc : {repr(vi_nfc)}")

    # Bước 2: clean_text
    en_clean = clean_text(en_raw)
    vi_clean = clean_text(vi_raw)
    print(f"   EN clean: {repr(en_clean)}")
    print(f"   VI clean: {repr(vi_clean)}")

    # Bước 3: is_valid_pair
    valid = is_valid_pair(en_clean, vi_clean)
    status = "✅ PASS" if valid else "❌ LOẠI"
    print(f"   is_valid_pair → {status}")
    print()

📌 [Bình thường]
   EN raw : 'Hello, how are you?'
   VI raw : 'Xin chào, bạn có khoẻ không?'
   EN nfc : 'Hello, how are you?'
   VI nfc : 'Xin chào, bạn có khoẻ không?'
   EN clean: 'Hello, how are you?'
   VI clean: 'Xin chào, bạn có khoẻ không?'
   is_valid_pair → ✅ PASS

📌 [Subtitle noise: [Muttering], ♪, --, CHAVEZ:]
   EN raw : '[Muttering] - Oh, I see ♪ ---- hello ----'
   VI raw : 'CHAVEZ: >> Ừ, tôi thấy rồi.'
   EN nfc : '[Muttering] - Oh, I see ♪ ---- hello ----'
   VI nfc : 'CHAVEZ: >> Ừ, tôi thấy rồi.'
   EN clean: 'Oh, I see hello'
   VI clean: 'Ừ, tôi thấy rồi.'
   is_valid_pair → ✅ PASS

📌 [HTML + URL]
   EN raw : 'Visit <b>our</b> site at https://example.com for more info.'
   VI raw : 'Truy cập trang web của <b>chúng tôi</b> tại www.example.vn.'
   EN nfc : 'Visit <b>our</b> site at https://example.com for more info.'
   VI nfc : 'Truy cập trang web của <b>chúng tôi</b> tại www.example.vn.'
   EN clean: 'Visit our site at for more info.'
   VI clean: 'Truy cập trang we

## 3. Kiểm tra thông số lọc của is_valid_pair

In [ ]:
# Kiểm tra biên là_valid_pair với các tham số khác nhau
cases = [
    {"desc": "Câu quá ngắn (1 từ)",          "en": "Hi",       "vi": "Xin chào"},
    {"desc": "Câu đúng biên min_len=1",        "en": "Go",      "vi": "Đi thôi"},
    {"desc": "Tỉ lệ độ dài vượt max_len_ratio","en": "Go now",  "vi": " ".join(["đi"] * 30)},
    {"desc": "Alpha ratio quá thấp (<40%)",    "en": "12345 6789 ###", "vi": "123 456 789"},
]

for c in cases:
    v = is_valid_pair(c["en"], c["vi"])
    print(f"{'✅' if v else '❌'}  {c['desc']}")
    print(f"      en={repr(c['en'][:60])}")
    print(f"      vi={repr(c['vi'][:60])}")
    print()

## 4. Load file CSV thực tế qua load_dataset()

> Chỉ load file nhỏ để demo. Đổi path nếu cần.

In [ ]:
import os

# Kiểm tra xem file tồn tại không
SAMPLE_CSV = "../data/opus100/train.csv"   # đổi nếu cần

if not os.path.exists(SAMPLE_CSV):
    print(f"⚠️  File không tồn tại: {SAMPLE_CSV}")
    print("   → Tạo file CSV mẫu để test...")

    import pandas as pd
    os.makedirs("../data/sample", exist_ok=True)
    sample_data = {
        "en": [
            "Hello, how are you?",
            "The weather is nice today.",
            "[Music playing] - Good morning!",
            "Visit https://example.com for more.",
            "OK",  # sẽ bị lọc (en == vi)
            "I love learning new languages every day.",
            "She is reading a book about history.",
        ],
        "vi": [
            "Xin chào, bạn có khoẻ không?",
            "Thời tiết hôm nay rất đẹp.",
            "Chào buổi sáng!",
            "Truy cập website của chúng tôi để biết thêm.",
            "OK",  # sẽ bị lọc
            "Tôi yêu thích việc học ngôn ngữ mới mỗi ngày.",
            "Cô ấy đang đọc một cuốn sách về lịch sử.",
        ]
    }
    df_sample = pd.DataFrame(sample_data)
    df_sample.to_csv("../data/sample/sample.csv", index=False)
    SAMPLE_CSV = "../data/sample/sample.csv"
    print(f"   → Đã tạo: {SAMPLE_CSV}")

df = load_dataset(SAMPLE_CSV, max_len=200)
print(f"\nKết quả: {len(df)} cặp sạch")
df.head(10)

## 5. TranslationDataset — Tokenize 1 cặp câu thành tensor

In [ ]:
import os
from src.data.dataset import TranslationDataset, collate_fn

SPM_PATH = "../models/spm_tokenizer.model"   # đổi nếu cần

if not os.path.exists(SPM_PATH):
    print(f"⚠️  Tokenizer chưa có: {SPM_PATH}")
    print("   → Bỏ qua cell này, chạy tokenizer.py trước.")
else:
    ds = TranslationDataset(
        data_path=SAMPLE_CSV,
        spm_model_path=SPM_PATH,
        max_len=128,
    )

    print(f"Dataset size: {len(ds)}")

    # Lấy phần tử đầu tiên
    sample = ds[0]
    print(f"\n--- Sample[0] ---")
    print(f"src tensor shape : {sample['src'].shape}")
    print(f"tgt tensor shape : {sample['tgt'].shape}")
    print(f"src token ids    : {sample['src'].tolist()}")
    print(f"tgt token ids    : {sample['tgt'].tolist()}")

    # Decode lại để kiểm tra
    import sentencepiece as spm
    sp = spm.SentencePieceProcessor()
    sp.load(SPM_PATH)

    src_tokens = sample['src'].tolist()
    tgt_tokens = sample['tgt'].tolist()
    # bỏ BOS/EOS khi decode
    src_decoded = sp.decode(src_tokens[1:-1])
    tgt_decoded = sp.decode(tgt_tokens[1:-1])

    print(f"\nEN decoded: {src_decoded}")
    print(f"VI decoded: {tgt_decoded}")

## 6. collate_fn — Batch & Padding

In [ ]:
import torch
from torch.utils.data import DataLoader

if not os.path.exists(SPM_PATH):
    print(f"⚠️  Bỏ qua: tokenizer chưa có.")
else:
    pad_id = ds.pad_id

    loader = DataLoader(
        ds,
        batch_size=4,
        shuffle=False,
        collate_fn=lambda x: collate_fn(x, pad_id=pad_id),
    )

    batch = next(iter(loader))

    print("=" * 50)
    print("Batch từ DataLoader:")
    print(f"  src shape : {batch['src'].shape}   (B x T_src)")
    print(f"  tgt shape : {batch['tgt'].shape}   (B x T_tgt)")
    print(f"  pad_id    : {pad_id}")
    print()

    print("src (raw IDs):")
    print(batch['src'])
    print()
    print("tgt_input  = tgt[:, :-1]:")
    print(batch['tgt'][:, :-1])
    print()
    print("tgt_expected = tgt[:, 1:]:")
    print(batch['tgt'][:, 1:])

    # Kiểm tra padding
    pad_mask_src = (batch['src'] == pad_id)
    print(f"\nSố token PAD trong src : {pad_mask_src.sum().item()}")
    print(f"Số token PAD trong tgt : {(batch['tgt'] == pad_id).sum().item()}")

## 7. Tóm tắt pipeline cho 1 cặp câu

In [ ]:
# Chạy toàn bộ pipeline cho 1 cặp câu mẫu
EN = "[Music] - Hello, visit https://example.com — how are you doing?"
VI = ">> Xin chào, truy cập website — bạn có khoẻ không?"

print("🚀 Pipeline cho 1 cặp câu")
print("-" * 60)

# Bước 1
en1 = normalize_unicode(EN)
vi1 = normalize_unicode(VI)
print(f"[1] normalize_unicode")
print(f"    EN: {repr(en1)}")
print(f"    VI: {repr(vi1)}")

# Bước 2
en2 = clean_text(EN)
vi2 = clean_text(VI)
print(f"\n[2] clean_text")
print(f"    EN: {repr(en2)}")
print(f"    VI: {repr(vi2)}")

# Bước 3
valid = is_valid_pair(en2, vi2)
print(f"\n[3] is_valid_pair → {'✅ PASS' if valid else '❌ LOẠI'}")

if valid and os.path.exists(SPM_PATH):
    # Bước 4: Tokenize
    import sentencepiece as spm
    sp = spm.SentencePieceProcessor()
    sp.load(SPM_PATH)

    bos_id = sp.bos_id()
    eos_id = sp.eos_id()

    en_ids = [bos_id] + sp.encode(en2)[:126] + [eos_id]
    vi_ids = [bos_id] + sp.encode(vi2)[:126] + [eos_id]

    print(f"\n[4] Tokenize (SPM)")
    print(f"    EN IDs ({len(en_ids)} tokens): {en_ids}")
    print(f"    VI IDs ({len(vi_ids)} tokens): {vi_ids}")

    # Bước 5: Tensor
    src_t = torch.tensor([en_ids])
    tgt_t = torch.tensor([vi_ids])
    print(f"\n[5] Tensor")
    print(f"    src: {src_t.shape} → {src_t}")
    print(f"    tgt: {tgt_t.shape} → {tgt_t}")

    # Bước 6: Teacher forcing split
    tgt_in  = tgt_t[:, :-1]   # decoder input  (BOS ... last-1)
    tgt_out = tgt_t[:, 1:]    # decoder target (1 ... EOS)
    print(f"\n[6] Teacher forcing split")
    print(f"    tgt_input    : {tgt_in}")
    print(f"    tgt_expected : {tgt_out}")
elif valid and not os.path.exists(SPM_PATH):
    print("   ⚠️  Tokenizer chưa có → bỏ qua bước 4-6")

print("\n✅ Pipeline hoàn tất!")